In [2]:
import json

with open("../data/raw/cbi_b31_raw.json", "r", encoding="utf-8") as file:
    data = json.load(file)

print(data.keys())
print(data["result"].keys())

dict_keys(['help', 'success', 'result'])
dict_keys(['include_total', 'limit', 'records_format', 'resource_id', 'total_estimation_threshold', 'records', 'fields', '_links', 'total', 'total_was_estimated'])


In [3]:
print("Number of records:", len(data["result"]["records"]))

print("\nFirst record:")
print(data["result"]["records"][0])

Number of records: 47

First record:
{'_id': 1, 'Reporting date': '2014-12-31T00:00:00', 'buy_to_let_properties__floating_rate__standard_or_ltv_variab': 4.53, 'buy_to_let_properties__floating_rate__tracker_mortgages__rat': 1.09, 'buy_to_let_properties__fixed_rate__from_1_to_3_years__rates_': 5.33, 'buy_to_let_properties__fixed_rate__over_3_years__rates_on_ou': 4.54, 'principal_dwelling_houses__floating_rate__standard_or_ltv_va': 4.19, 'principal_dwelling_houses__floating_rate__tracker_mortgages_': 1.03, 'principal_dwelling_houses__floating_rate__up_to_1_year_fixed': 3.33, 'principal_dwelling_houses__fixed_rate__from_1_to_3_years__ra': 4.49, 'principal_dwelling_houses__fixed_rate__over_3_years__rates_o': 3.94, 'buy_to_let_properties__floating_rate__standard_or_ltv_variab_2': '5473', 'buy_to_let_properties__floating_rate__tracker_mortgages__out': '12333', 'buy_to_let_properties__fixed_rate__from_1_to_3_years__outsta': 52, 'buy_to_let_properties__fixed_rate__over_3_years__outstanding': 16

In [4]:
print("\nFields:")
print(data["result"]["fields"])


Fields:
[{'id': '_id', 'type': 'int'}, {'id': 'Reporting date', 'type': 'timestamp', 'info': {'label': 'Reporting date'}}, {'id': 'buy_to_let_properties__floating_rate__standard_or_ltv_variab', 'type': 'numeric', 'info': {'label': 'Buy-to-let properties, floating rate, standard or LTV variable, rates on outstanding amounts (%)'}}, {'id': 'buy_to_let_properties__floating_rate__tracker_mortgages__rat', 'type': 'numeric', 'info': {'label': 'Buy-to-let properties, floating rate, tracker mortgages, rates on outstanding amounts (%)'}}, {'id': 'buy_to_let_properties__fixed_rate__from_1_to_3_years__rates_', 'type': 'numeric', 'info': {'label': 'Buy-to-let properties, fixed rate, from 1 to 3 years, rates on outstanding amounts (%)'}}, {'id': 'buy_to_let_properties__fixed_rate__over_3_years__rates_on_ou', 'type': 'numeric', 'info': {'label': 'Buy-to-let properties, fixed rate, over 3 years, rates on outstanding amounts (%)'}}, {'id': 'principal_dwelling_houses__floating_rate__standard_or_ltv_va

this gives us everything we need. The B.3.1 API is returning 47 quarterly observations with several mortgage-rate categories.

For our MVP, I recommend we focus on Principal Dwelling Houses (PDH) because that's the most relevant series for the household/mortgage story.

Rather than choosing one mortgage type arbitrarily, let's start with the PDH floating rate — standard or LTV variable, rates on outstanding amounts:

Principal Dwelling Houses, floating rate,
standard or LTV variable,
rates on outstanding amounts (%)


# create the processed dataframe


In [5]:
import pandas as pd

records = data["result"]["records"]

mortgage_rates = pd.DataFrame(records)

mortgage_rates = mortgage_rates[
    [
        "Reporting date",
        "principal_dwelling_houses__floating_rate__standard_or_ltv_va"
    ]
].copy()

mortgage_rates = mortgage_rates.rename(columns={
    "Reporting date": "date",
    "principal_dwelling_houses__floating_rate__standard_or_ltv_va":
        "mortgage_rate"
})

mortgage_rates["date"] = pd.to_datetime(mortgage_rates["date"])

mortgage_rates = mortgage_rates.sort_values("date").reset_index(drop=True)

print(mortgage_rates.head())
print(mortgage_rates.tail())
print("Shape:", mortgage_rates.shape)

        date  mortgage_rate
0 2014-12-31           4.19
1 2015-03-31           4.26
2 2015-06-30           4.11
3 2015-09-30           4.08
4 2015-12-31           3.97
         date  mortgage_rate
42 2025-06-30           4.15
43 2025-09-30           4.14
44 2025-12-31           4.13
45 2026-03-31           4.12
46 2026-06-30           4.13
Shape: (47, 2)


# validate

In [6]:
print("Missing values:")
print(mortgage_rates.isna().sum())

print("\nDuplicate dates:")
print(mortgage_rates["date"].duplicated().sum())

print("\nDate range:")
print(
    mortgage_rates["date"].min(),
    "to",
    mortgage_rates["date"].max()
)

print("\nMortgage rate range:")
print(
    mortgage_rates["mortgage_rate"].min(),
    "to",
    mortgage_rates["mortgage_rate"].max()
)

Missing values:
date             0
mortgage_rate    0
dtype: int64

Duplicate dates:
0

Date range:
2014-12-31 00:00:00 to 2026-06-30 00:00:00

Mortgage rate range:
3.43 to 4.26


# Save it!

In [8]:
mortgage_rates.to_csv(
    "../data/processed/mortgage_rates.csv",
    index=False
)

print("Saved to data/processed/mortgage_rates.csv")

Saved to data/processed/mortgage_rates.csv
